# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The metadata object is a structured metadata, not a dictionary.
metadata_obj = dataset.metadata

# Access title and description
print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each entity in the dataset, such as record sets, fields, and columns, is referenced by its `@id`. Let's list the available record sets.

In [ ]:
# List all available record sets and their @id
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in the metadata. Please check dataset schema.")
else:
    print("Available Record Sets:")
    for recset in record_sets:
        print(f"- @id: {recset['@id']} | name: {recset.get('name','')} | description: {recset.get('description','')}")

# For each record set, show fields and columns referenced by @id
for recset in record_sets:
    print(f"\nRecord Set '@id': {recset['@id']} Fields:")
    fields = recset.get('field', recset.get('fields', []))
    if fields:
        for f in fields:
            if isinstance(f, dict):
                field_id = f.get('@id', str(f))
                name = f.get('name','')
                print(f"-- field @id: {field_id} | name: {name}")
            else:
                print(f"-- field @id: {f}")
    else:
        print("-- No fields defined.")

    # Show columns if available
    columns = recset.get('column', recset.get('columns', []))
    if columns:
        print("Columns:")
        for col in columns:
            if isinstance(col, dict):
                print(f"-- column @id: {col.get('@id',str(col))} | name: {col.get('name','')}")
            else:
                print(f"-- column @id: {col}")
    else:
        print("-- No columns defined.")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, we'll extract all available record sets

record_set_ids = [recset['@id'] for recset in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for Record Set @id: {record_set_id}, shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Display first few rows from one record set
if dataframes:
    first_recset = next(iter(dataframes.keys()))
    print(f"\nPreview of data for record set @id: {first_recset}")
    display(dataframes[first_recset].head())
else:
    print("No DataFrames loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All references to entities use their `@id` values.

In [ ]:
# Choose the main record set for analysis
main_record_set_id = None
if record_set_ids:
    # Use the first as example
    main_record_set_id = record_set_ids[0]
else:
    raise RuntimeError("No record set IDs found.")

df = dataframes[main_record_set_id]

# List numeric columns by @id
numeric_columns = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric columns for @id {main_record_set_id}: {numeric_columns}")

if numeric_columns:
    # Use first numeric column as example
    numeric_field_id = numeric_columns[0]
else:
    print("No numeric columns found.")
    numeric_field_id = None

threshold = 10

if numeric_field_id:
    # Filter records
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where field @id '{numeric_field_id}' > {threshold}:")
    display(filtered_df.head())

    # Normalize the values
    field_norm_col = f"{numeric_field_id}_normalized"
    filtered_df[field_norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, field_norm_col]].head())

    # Choose a group field by @id (for demo, pick a non-numeric field if available)
    group_field_candidates = [col for col in df.columns if col != numeric_field_id]
    group_field = None
    for col in group_field_candidates:
        if df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col]):
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by field @id '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot histograms and pairwise relationships for numeric fields, referencing fields using their `@id`.

In [ ]:
# Visualization of numeric field distribution
if numeric_field_id and not filtered_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True, ax=ax)
    ax.set_title(f"Distribution of '{numeric_field_id}' (filtered > {threshold})")
    ax.set_xlabel(f"Field @id: {numeric_field_id}")
    plt.show()

    # Pairplot of numeric fields if more than one
    if len(numeric_columns) > 1:
        sns.pairplot(filtered_df[numeric_columns])
        plt.suptitle("Pairwise Relationships Among Numeric Fields")
        plt.show()

    # Bar plot by group field if available
    if group_field:
        plt.figure(figsize=(8,5))
        group_means = filtered_df.groupby(group_field)[numeric_field_id].mean()
        group_means.plot(kind='bar')
        plt.title(f"Mean of '{numeric_field_id}' by '{group_field}'")
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Not enough data available for numeric visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded FAIR² mental health survey data using `mlcroissant`.
- Used entity `@id` for consistent referencing of record sets, fields, and columns.
- Performed basic filtering, normalization, and grouping on numeric fields.
- Visualizations highlighted distributions and relationships in the sample data.
- This process demonstrates the structured approach enabled by Croissant schema for robust, reproducible research data analysis in Python.